# Data Extraction — Finnish Target Domain (DGT × Books)

This notebook prepares all data needed: target domain **𝑙 = Finnish**, **𝑔 = legal/administrative (OPUS DGT)**, **¬𝑔 = literary (OPUS Books)**.

It produces three corpora on disk under `data/`:

| Corpus | Source | Content | Used for |
|---|---|---|---|
| `target_domain_fi_dgt/` | `opus_dgt`, Finnish only | the intersection domain itself | train/val/test split, final evaluation  |
| `genre_module_dgt_non_fi/` | `opus_dgt`, all languages **except** Finnish (𝐿∖𝑙) | legal/admin text in many languages | training the **genre module** 𝑔  |
| `language_module_books_fi/` | `opus_books`, Finnish only (𝑙 portion of ¬𝑔) | literary Finnish | training the **language module** 𝑙  |

No modeling happens here — this notebook only extracts, cleans, splits, and saves data that a
later "module training" notebook will consume.


## 0. Setup


In [ ]:
import json
import random
from collections import defaultdict
from pathlib import Path

import pandas as pd
from datasets import load_dataset

SEED = 42
RNG = random.Random(SEED)

DATA_DIR = Path("data")
# three separate output dirs 
TARGET_DIR = DATA_DIR / "target_domain_fi_dgt"      # l ∩ g: Finnish DGT (train/val/test)
GENRE_DIR = DATA_DIR / "genre_module_dgt_non_fi"     # L∖l: DGT, every language except Finnish
LANG_DIR = DATA_DIR / "language_module_books_fi"     # ¬g portion of l: Finnish Books
for d in (TARGET_DIR, GENRE_DIR, LANG_DIR):
    d.mkdir(parents=True, exist_ok=True)

TARGET_LANG = "fi"

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. DGT as multi-parallel corpus



In [2]:
from datasets import get_dataset_config_names

# opus_dgt/opus_books are stored as one HF "config" per language PAIR (e.g. "en-fi"), not one
# config per language -- so every sentence in a target language is scattered across several
# configs and has to be collected by iterating all pairs that include it (see collect_sentences)
dgt_configs_available = get_dataset_config_names("Helsinki-NLP/opus_dgt")
print(f"opus_dgt: {len(dgt_configs_available)} pairs available via load_dataset:")
print(dgt_configs_available)

books_configs_available = get_dataset_config_names("Helsinki-NLP/opus_books")
books_fi_configs = sorted(c for c in books_configs_available if TARGET_LANG in c.split("-"))
print(f"\nopus_books: {len(books_configs_available)} pairs available, {len(books_fi_configs)} involve Finnish:")
print(books_fi_configs)

opus_dgt: 10 pairs available via load_dataset:
['bg-ga', 'bg-hr', 'bg-sh', 'es-ga', 'fi-ga', 'ga-nl', 'ga-sh', 'hr-sk', 'hr-sv', 'mt-sh']



opus_books: 64 pairs available, 6 involve Finnish:
['en-fi', 'es-fi', 'fi-fr', 'fi-hu', 'fi-no', 'fi-pl']


In [3]:
DGT_CONFIGS = dgt_configs_available          # all 10 pre-converted opus_dgt pairs
BOOKS_FI_CONFIGS = books_fi_configs          # all opus_books pairs involving Finnish

## 2. Helpers

Light clean up; Deduplication with preservation of first-see order

In [ ]:
MIN_TOKENS = 3  # drop lines shorter than this (whitespace tokens)

def is_clean(text: str) -> bool:
    if not text:
        return False
    if text.lower().startswith("source:"):
        return False  # DGT sometimes leaves citation stubs like "Source: ..." as their own "sentence"
    return len(text.split()) >= MIN_TOKENS

def dedup_keep_order(lines):
    # dict.fromkeys preserves first-seen order while dropping exact-duplicate sentences
    return list(dict.fromkeys(lines))

def whitespace_tokens(text: str) -> int:
    return len(text.split())

def total_tokens(lines) -> int:
    return sum(whitespace_tokens(l) for l in lines)

def collect_sentences(dataset_name, configs, keep_lang=None, exclude_lang=None):
    """Stream all configs, bucket cleaned sentences per language, dedup per language.

    keep_lang / exclude_lang let one function serve three different needs in this notebook:
    - keep_lang="fi"      -> only the target language (target domain, language-module corpus)
    - exclude_lang="fi"   -> every language except the target one (genre-module corpus, L∖l)
    - neither             -> everything (not used directly here, kept for generality)
    """
    per_lang = defaultdict(list)
    seen = defaultdict(set)  # per-language dedup set, since the same sentence can recur across pairs
    for cfg in configs:
        l1, l2 = cfg.split("-")
        ds = load_dataset(dataset_name, cfg, split="train")
        for row in ds:
            tr = row["translation"]
            for lang in (l1, l2):
                if exclude_lang and lang == exclude_lang:
                    continue
                if keep_lang and lang != keep_lang:
                    continue
                text = tr[lang].strip()
                if not is_clean(text) or text in seen[lang]:
                    continue
                seen[lang].add(text)
                per_lang[lang].append(text)
    return per_lang

## 3. Target domain: Finnish DGT (𝑙 ∩ 𝑔)

This is the low-resource intersection domain itself usable for the lightweight supervised adaptation step.


In [ ]:
# l ∩ g: Finnish DGT sentences: Ttarget domain 
target_per_lang = collect_sentences("Helsinki-NLP/opus_dgt", DGT_CONFIGS, keep_lang=TARGET_LANG)
target_fi_sentences = target_per_lang[TARGET_LANG]

print(f"Unique cleaned Finnish DGT sentences: {len(target_fi_sentences):,}")
print(f"Total whitespace tokens available:    {total_tokens(target_fi_sentences):,}")

Unique cleaned Finnish DGT sentences: 135,788
Total whitespace tokens available:    2,581,936


### 3.1 Train / validation / test split

Training portion capped at 1,000,000 whitespace tokens. The dataset is shuffled, thene train, val e test set extracted. Any leftover sentences are left unused. 


In [ ]:
TRAIN_TOKEN_CAP = 1_000_000  # train portion may contain at most 1M
                              # whitespace tokens, to simulate a low-resource intersection domain
VAL_TOKEN_CAP = 50_000
TEST_TOKEN_CAP = 50_000

def split_by_token_budget(lines, caps, seed=SEED):
    """Fills each split up to its token budget (not a sentence-count budget)"""
    rng = random.Random(seed)
    pool = lines[:]
    rng.shuffle(pool)
    pool = list(reversed(pool))  # so we can .pop() from the "front" in shuffled order

    splits = {}
    for name, cap in caps.items():
        out, tok = [], 0
        while pool and tok < cap:
            ln = pool[-1]
            n = whitespace_tokens(ln)
            if tok + n > cap and out:
                break  # stop before overshooting the cap, unless the split would otherwise be empty
            out.append(pool.pop())
            tok += n
        splits[name] = out
    return splits, pool  # pool = leftover, unused sentences

target_splits, target_unused = split_by_token_budget(
    target_fi_sentences,
    {"train": TRAIN_TOKEN_CAP, "val": VAL_TOKEN_CAP, "test": TEST_TOKEN_CAP},
)

for name, lines in target_splits.items():
    print(f"{name:5s}: {len(lines):>7,} sentences, {total_tokens(lines):>9,} tokens")
print(f"unused: {len(target_unused):>7,} sentences (left in the pool, not written to disk)")

train:  52,509 sentences,   999,994 tokens
val  :   2,630 sentences,    49,968 tokens
test :   2,618 sentences,    49,997 tokens
unused:  78,031 sentences (left in the pool, not written to disk)


In [ ]:
# one JSONL file per split, {"text": ...} per line
for name, lines in target_splits.items():
    out_path = TARGET_DIR / f"{name}.jsonl"
    with out_path.open("w", encoding="utf-8") as f:
        for text in lines:
            f.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")
    print(f"wrote {out_path} ({len(lines):,} lines)")

wrote data/target_domain_fi_dgt/train.jsonl (52,509 lines)
wrote data/target_domain_fi_dgt/val.jsonl (2,630 lines)
wrote data/target_domain_fi_dgt/test.jsonl (2,618 lines)


## 4. Genre module data: OPUS DGT, all languages except Finnish (𝐿∖𝑙)


In [ ]:
# L∖l: every DGT language except Finnish: the genre module's training data
genre_per_lang = collect_sentences("Helsinki-NLP/opus_dgt", DGT_CONFIGS, exclude_lang=TARGET_LANG)

genre_stats = pd.DataFrame(
    [(lang, len(sents), total_tokens(sents)) for lang, sents in genre_per_lang.items()],
    columns=["lang", "n_sentences", "n_tokens"],
).sort_values("n_sentences", ascending=False).reset_index(drop=True)
genre_stats

,lang,n_sentences,n_tokens
0,bg,1299968,30507633
1,mt,956777,18659132
2,sh,922588,18215248
3,hr,462156,9082524
4,sk,453887,9251943
5,sv,443266,9237188
6,es,142036,4323702
7,ga,141037,4220859
8,nl,133727,3659179


### 4.1 Balance across languages

Raw per-language counts are wildly uneven (up to ~9x between the largest and smallest language). Each language is capped at `MAX_SENTENCES_PER_LANG` so no single language dominates the genre-module training signal, and so total compute stays bounded.


In [9]:
MAX_SENTENCES_PER_LANG = 50_000  # without this, a handful of high-resource languages would
                                  # dominate the genre module's training signal

genre_rows = []
for lang, sents in genre_per_lang.items():
    pool = sents[:]
    RNG.shuffle(pool)
    kept = pool[:MAX_SENTENCES_PER_LANG]
    for text in kept:
        genre_rows.append({"lang": lang, "text": text})

genre_df = pd.DataFrame(genre_rows)
print(f"Total genre-module sentences after per-language cap: {len(genre_df):,}")
print(f"Total whitespace tokens: {genre_df['text'].map(whitespace_tokens).sum():,}")
genre_df.groupby("lang").size().sort_values(ascending=False)

Total genre-module sentences after per-language cap: 450,000


Total whitespace tokens: 10,569,868


lang
bg    50000
es    50000
ga    50000
hr    50000
mt    50000
nl    50000
sh    50000
sk    50000
sv    50000
dtype: int64

### 4.2 Train / dev split

No token cap applies here (only the target-domain training portion is capped), but a small, fixed-size dev slice per language for monitoring the genre module's training loss is held out.


In [ ]:
DEV_PER_LANG = 500  # fixed per-language dev slice, so no single language dominates the dev set either

train_rows, dev_rows = [], []
for lang, group in genre_df.groupby("lang"):
    idx = group.index.tolist()
    RNG.shuffle(idx)
    dev_idx, train_idx = set(idx[:DEV_PER_LANG]), set(idx[DEV_PER_LANG:])
    train_rows.append(group.loc[list(train_idx)])
    dev_rows.append(group.loc[list(dev_idx)])

genre_train_df = pd.concat(train_rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
genre_dev_df = pd.concat(dev_rows).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"genre train: {len(genre_train_df):,} rows | genre dev: {len(genre_dev_df):,} rows")

# the genre and language module corpora are deliberately left "text-rich(er)"
for name, df in [("train", genre_train_df), ("dev", genre_dev_df)]:
    out_path = GENRE_DIR / f"{name}.jsonl"
    with out_path.open("w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            f.write(json.dumps({"text": row["text"], "lang": row["lang"]}, ensure_ascii=False) + "\n")
    print(f"wrote {out_path} ({len(df):,} lines)")

genre train: 445,500 rows | genre dev: 4,500 rows


wrote data/genre_module_dgt_non_fi/train.jsonl (445,500 lines)
wrote data/genre_module_dgt_non_fi/dev.jsonl (4,500 lines)


## 5. Language module data: OPUS Books, Finnish only (𝑙 portion of ¬𝑔)


In [ ]:
# ¬g portion of l: Finnish OPUS Books (literary), the language module's training data
books_per_lang = collect_sentences("Helsinki-NLP/opus_books", BOOKS_FI_CONFIGS, keep_lang=TARGET_LANG)
books_fi_sentences = books_per_lang[TARGET_LANG]

print(f"Unique cleaned Finnish Books sentences: {len(books_fi_sentences):,}")
print(f"Total whitespace tokens available:      {total_tokens(books_fi_sentences):,}")

Unique cleaned Finnish Books sentences: 4,485
Total whitespace tokens available:      67,417


This is far smaller than the DGT Finnish pool (tens of thousands of tokens vs. millions) — OPUS
Books is inherently a small, curated literary corpus, and the six Finnish-involving pairs overlap
heavily (they're largely the same handful of translated novels aligned against different pivot
languages)

Small fixed dev slice (not token-capped, since this pool is already small) for
monitoring training loss; everything else goes to train.


In [12]:
BOOKS_DEV_SIZE = 200  # this corpus is tiny to begin with (a few thousand sentences), so the
                      # dev slice is a fixed small count rather than a token budget

pool = books_fi_sentences[:]
RNG.shuffle(pool)
books_dev = pool[:BOOKS_DEV_SIZE]
books_train = pool[BOOKS_DEV_SIZE:]

print(f"books train: {len(books_train):,} sentences, {total_tokens(books_train):,} tokens")
print(f"books dev:   {len(books_dev):,} sentences, {total_tokens(books_dev):,} tokens")

for name, lines in [("train", books_train), ("dev", books_dev)]:
    out_path = LANG_DIR / f"{name}.jsonl"
    with out_path.open("w", encoding="utf-8") as f:
        for text in lines:
            f.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")
    print(f"wrote {out_path} ({len(lines):,} lines)")

books train: 4,285 sentences, 64,358 tokens
books dev:   200 sentences, 3,059 tokens
wrote data/language_module_books_fi/train.jsonl (4,285 lines)
wrote data/language_module_books_fi/dev.jsonl (200 lines)


## 6. Sanity checks

Quick checks worth keeping: (a) no leakage between the Finnish DGT target domain and the Finnish
Books language-module corpus (different source datasets, but worth confirming no coincidental
overlap — e.g. boilerplate), and (b) that the target-domain splits are disjoint from each other.


In [ ]:
# leakage check: the language module (Finnish Books) and the target domain (Finnish DGT) are
# different genres of the same language, so they shouldn't share sentences -- but worth confirming
# rather than assuming, since a leak here would silently inflate the language module's zero-shot
# transfer numbers
target_all = set(target_splits["train"]) | set(target_splits["val"]) | set(target_splits["test"])
books_all = set(books_train) | set(books_dev)

overlap_target_books = target_all & books_all
print(f"Overlap between target-domain (DGT fi) and language-module (Books fi): {len(overlap_target_books)} sentences")

assert not (set(target_splits["train"]) & set(target_splits["val"]))
assert not (set(target_splits["train"]) & set(target_splits["test"]))
assert not (set(target_splits["val"]) & set(target_splits["test"]))
print("target-domain train/val/test are pairwise disjoint: OK")

Overlap between target-domain (DGT fi) and language-module (Books fi): 0 sentences
target-domain train/val/test are pairwise disjoint: OK


## 7. Summary


In [ ]:
# final tally of every corpus this project trains/evaluates on
summary = pd.DataFrame([
    {"corpus": "target_domain_fi_dgt/train", "sentences": len(target_splits["train"]), "tokens": total_tokens(target_splits["train"])},
    {"corpus": "target_domain_fi_dgt/val",   "sentences": len(target_splits["val"]),   "tokens": total_tokens(target_splits["val"])},
    {"corpus": "target_domain_fi_dgt/test",  "sentences": len(target_splits["test"]),  "tokens": total_tokens(target_splits["test"])},
    {"corpus": "genre_module_dgt_non_fi/train", "sentences": len(genre_train_df), "tokens": int(genre_train_df["text"].map(whitespace_tokens).sum())},
    {"corpus": "genre_module_dgt_non_fi/dev",   "sentences": len(genre_dev_df),   "tokens": int(genre_dev_df["text"].map(whitespace_tokens).sum())},
    {"corpus": "language_module_books_fi/train", "sentences": len(books_train), "tokens": total_tokens(books_train)},
    {"corpus": "language_module_books_fi/dev",   "sentences": len(books_dev),   "tokens": total_tokens(books_dev)},
])
summary

,corpus,sentences,tokens
0,target_domain_fi_dgt/train,52509,999994
1,target_domain_fi_dgt/val,2630,49968
2,target_domain_fi_dgt/test,2618,49997
3,genre_module_dgt_non_fi/train,445500,10465919
4,genre_module_dgt_non_fi/dev,4500,103949
5,language_module_books_fi/train,4285,64358
6,language_module_books_fi/dev,200,3059
